In [1]:
from vllm_utils import *
from sft import *
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig

INFO 03-03 13:48:01 __init__.py:190] Automatically detected platform cuda.


/root/autodl-tmp/assignment5-alignment/cs336_alignment/sft.py:100: SyntaxWarning: invalid escape sequence '\('
  - "log_probs":形状为（batch_size, sequence_length）,条件对数概率\(log p_{\theta}(x_t | x_{<<t})\)。


In [2]:
model_name ='../model/Qwen2.5-Math-1.5B'
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="cuda",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
vllm = init_vllm(model_name,device='cuda',seed=42)

INFO 03-03 13:48:15 config.py:542] This model supports multiple tasks: {'embed', 'score', 'reward', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 03-03 13:48:15 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, num_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-03 13:48:17 model_runner.py:1115] Loading model weights took 2.8789 GB
INFO 03-03 13:48:18 worker.py:267] Memory profiling takes 0.73 seconds
INFO 03-03 13:48:18 worker.py:267] the current vLLM instance can use total_gpu_memory (31.48GiB) x gpu_memory_utilization (0.85) = 26.76GiB
INFO 03-03 13:48:18 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 22.43GiB.
INFO 03-03 13:48:19 executor_base.py:110] # CUDA blocks: 52500, # CPU blocks: 9362
INFO 03-03 13:48:19 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 205.08x
INFO 03-03 13:48:22 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  1.99it/s]

INFO 03-03 13:48:39 model_runner.py:1562] Graph capturing finished in 18 secs, took 0.21 GiB
INFO 03-03 13:48:39 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 22.02 seconds


# 定义vllm基础函数
Logging generations in-the-loop
在模型训练循环中记录生成结果是良好的实践，监督微调（SFT）/强化学习（RL）场景也不例外。编写log_generations函数，用于让模型对给定提示词（如从验证集中采样的提示词）生成响应并记录日志。建议为每个示例至少记录以下内容：

+ 输入提示词 ✔
+ 监督微调（SFT）/强化学习（RL）模型生成的响应 ✔
+ 真实答案 ✔
+ 奖励信息，包括格式、答案及总奖励 ✔
+ 响应的平均token熵 ✔
+ 平均响应长度、正确响应的平均长度及错误响应的平均长度。


In [5]:
prompts = ['What is the capital of France?','What is the largest mammal?']
ground_truths = ['The capital of France is Paris.','The largest mammal is the blue whale.']
sampling_params = SamplingParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=1024,
            stop = ["</answer>"],
)

In [1]:
from drgrpo_grader import r1_zero_reward_fn
overview = log_generation(
    prompts,
    ground_truths,
    r1_zero_reward_fn,
    model,
    tokenizer,
    vllm,
    sampling_params,
)

NameError: name 'log_generation' is not defined

In [8]:
overview['rows']

[{'prompt': 'What is the capital of France?',
  'response': ' The capital of France is Paris. It is the largest city in France and the capital of the City and Guilds union.',
  'true_answer': 'The capital of France is Paris.',
  'total_reward': 0.0,
  'format_reward': 0.0,
  'answer_reward': 0.0,
  'is_correct': False,
  'response_length': 25,
  'avg_token_entropy': 0.949999988079071},
 {'prompt': 'What is the largest mammal?',
  'response': ' The largest mammal is the blue whale. The blue whale is a breast-eating mammal that can reach a maximum length of up to 20 meters (66 feet). It is a subspecies of Bysthoma areus, which is considered the largest and heaviest of all mammals on Earth. Blue whales are known to be found in the southern hemisphere, where they are also known as World Krill.',
  'true_answer': 'The largest mammal is the blue whale.',
  'total_reward': 0.0,
  'format_reward': 0.0,
  'answer_reward': 0.0,
  'is_correct': False,
  'response_length': 86,
  'avg_token_entropy

In [9]:
#TODO:定义一个打印函数,讲采样结果打印出来

# 评估函数

In [11]:
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop = ["</answer>"],
)

In [18]:
from drgrpo_grader import r1_zero_reward_fn
rewards = evaluate_vllm(
    vllm,
    prompts,
    ground_truths,
    sampling_params,
    r1_zero_reward_fn,
)

Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.90s/it, est. speed input: 3.70 toks/s, output: 156.26 toks/s]


In [20]:
rewards

[{'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0},
 {'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}]

In [22]:
overview = {
    "sample_size": len(rewards),
    "answer_correct": 0,
    "format_correct": 0,
    "total_correct": 0,
    "format_correct_but_answer_wrong": 0,
    "answer_correct_but_format_wrong": 0,
    "total_wrong": 0,
    "accuracy": 0.0,
    'wrong_rate': 0.0,
    'contradictory_samples': 0,
}

for reward_dict in rewards:
    # 统计格式正确
    if reward_dict['format_reward'] == 1.0:
        overview['format_correct'] += 1
        
    # 统计答案正确
    if reward_dict['answer_reward'] == 1.0:
        overview['answer_correct'] += 1
    
    # 统计完全正确
    if reward_dict['reward'] == 1.0:
        overview['total_correct'] += 1
    
    # 统计格式正确但答案错误
    if reward_dict['format_reward'] == 1.0 and reward_dict['answer_reward'] == 0.0:
        overview['format_correct_but_answer_wrong'] += 1
    
    # 统计答案正确但格式错误
    if reward_dict['answer_reward'] == 1.0 and reward_dict['format_reward'] == 0.0:
        overview['answer_correct_but_format_wrong'] += 1

    if reward_dict['format_reward'] == 0.0 and reward_dict['answer_reward'] == 0.0:
        overview['total_wrong'] += 1

    # 统计自相矛盾的样本数
    if reward_dict['reward']==1.0 and (reward_dict['format_reward'] == 0.0 or reward_dict['answer_reward'] == 0.0):
        overview['contradictory_samples'] += 1

# 计算准确率
if overview['sample_size'] > 0:
    overview['accuracy'] = overview['total_correct'] / overview['sample_size']
    overview['wrong_rate'] = overview['total_wrong'] / overview['sample_size']
    
print(overview)
    

{'sample_size': 2, 'answer_correct': 0, 'format_correct': 0, 'total_correct': 0, 'format_correct_but_answer_wrong': 0, 'answer_correct_but_format_wrong': 0, 'total_wrong': 2, 'accuracy': 0.0, 'wrong_rate': 1.0}
